# NHANES 2021–2023 — Bước 4: Data Cleaning & Preprocessing (bản nháp có kiểm soát)

**Thực hiện:** Bạn tự chạy trong Jupyter Notebook.  
**Bước này:** lọc cohort, kiểm tra nhãn ứng viên, đánh giá dữ liệu thiếu, xuất bảng để rà soát.  
**Không chạy `.fit()`, không train/test split, không `predict()` và không tạo mô hình.**

⚠️ `LBXGH >= 5.7` chỉ là **chỉ dấu HbA1c tăng**, **không phải chẩn đoán type 2 hay dự đoán nguy cơ mắc bệnh trong tương lai**. Mọi tỷ lệ dưới đây đều chưa áp dụng trọng số khảo sát. Không dùng bộ dữ liệu này cho mục đích lâm sàng.

**Tệp cần có trong `data/raw/`**: `DEMO_L.xpt`, `BMX_L.xpt`, `GHB_L.xpt`, `DIQ_L.xpt`.

Nguồn: [CDC GHB_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.htm), [CDC DEMO_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.htm), [CDC DIQ_L](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DIQ_L.htm), [NIDDK HbA1c](https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test).

## 1. Xác định thư mục dự án, phiên bản môi trường

Cấu trúc `03-ML-Research/` giữ nguyên từ Bước 2–3. Nếu chạy notebook từ thư mục con `notebooks/`, đoạn mã tự tìm thư mục dự án.

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / "data" / "raw").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Không thấy data/raw; hãy mở notebook từ 03-ML-Research/notebooks.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR = PROJECT_ROOT / "data" / "outputs" / "preprocessing"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0], "| Pandas:", pd.__version__)
print("Data raw:", RAW_DIR)
print("Sẽ lưu bảng tổng hợp ở:", OUT_DIR)

Python: 3.13.9 | Pandas: 3.0.0
Data raw: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\raw
Sẽ lưu bảng tổng hợp ở: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\preprocessing


## 2. Đọc lại bốn file XPT, kiểm tra khóa `SEQN`

Không sửa dữ liệu gốc. Nếu có trùng/mất `SEQN` thì dừng notebook, không tiếp tục ghép.

In [2]:
STEMS = ["DEMO_L", "BMX_L", "GHB_L", "DIQ_L"]

def locate_xpt(stem):
    for ext in (".xpt", ".XPT"):
        path = RAW_DIR / f"{stem}{ext}"
        if path.is_file():
            return path
    raise FileNotFoundError(f"Thiếu {stem}.xpt ở {RAW_DIR}")

frames = {stem: pd.read_sas(locate_xpt(stem), format="xport") for stem in STEMS}
for stem, df in frames.items():
    assert "SEQN" in df.columns, f"{stem} không có SEQN"
    assert df["SEQN"].notna().all() and df["SEQN"].is_unique, f"{stem}: SEQN không hợp lệ"
    print(stem, df.shape)

cols = {
    "DEMO_L": ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDEXPRG", "WTMEC2YR", "SDMVSTRA", "SDMVPSU"],
    "BMX_L": ["SEQN", "BMXBMI", "BMXWAIST"],
    "GHB_L": ["SEQN", "LBXGH", "WTPH2YR"],
    "DIQ_L": ["SEQN", "DIQ010", "DIQ160"],
}
for stem, required_cols in cols.items():
    missing_cols = sorted(set(required_cols) - set(frames[stem].columns))
    if missing_cols:
        raise KeyError(f"{stem} thiếu cột: {missing_cols}; xem lại XPT/codebook")

merged = frames["DEMO_L"][cols["DEMO_L"]].copy()
for stem in STEMS[1:]:
    merged = merged.merge(frames[stem][cols[stem]], on="SEQN", how="left", validate="one_to_one")
assert merged["SEQN"].is_unique
adults = merged.loc[merged["RIDAGEYR"] >= 20].copy()
print("Merged:", merged.shape, "| Adults >=20:", len(adults))
print("Có HbA1c:", int(adults["LBXGH"].notna().sum()))
print("BMI + HbA1c:", int(adults[["BMXBMI", "LBXGH"]].notna().all(axis=1).sum()))

DEMO_L (11933, 27)
BMX_L (8860, 22)
GHB_L (7199, 3)
DIQ_L (11744, 9)
Merged: (11933, 13) | Adults >=20: 7809
Có HbA1c: 5767
BMI + HbA1c: 5686


## 3. Lọc quan sát có nhãn xét nghiệm và phân bố tham khảo HbA1c

Người thiếu HbA1c **không được tự gán giá trị nhãn**. Dùng hai ngưỡng 5,7% và 6,5% để mô tả vùng đo; điều này **không đồng nghĩa chẩn đoán**. Nên kiểm tra lại theo mục đích nghiên cứu và tiêu chuẩn lâm sàng phù hợp trước khi dùng cho mô hình.

In [3]:
all_measured = adults.loc[adults["LBXGH"].notna()].copy()
all_measured["a1c_ge_57_candidate"] = (all_measured["LBXGH"] >= 5.7).astype("int64")
all_measured["a1c_band_review"] = pd.cut(
    all_measured["LBXGH"], bins=[-np.inf, 5.7, 6.5, np.inf],
    labels=["<5.7", "5.7-<6.5", ">=6.5"], right=False
)
print("N có HbA1c:", len(all_measured))
display(all_measured["a1c_band_review"].value_counts(sort=False).to_frame("n"))
print("Nhãn ứng viên 0/1 (CHƯA PHẢI CHẨN ĐOÁN):")
display(all_measured["a1c_ge_57_candidate"].value_counts(sort=False).to_frame("n"))

N có HbA1c: 5767


,n
a1c_band_review,
<5.7,3468
5.7-<6.5,1578
>=6.5,721


Nhãn ứng viên 0/1 (CHƯA PHẢI CHẨN ĐOÁN):


,n
a1c_ge_57_candidate,
0,3468
1,2299


## 4. So sánh ba quần thể nghiên cứu (chưa chốt tập chính)

- `all_measured`: tất cả người >=20 có HbA1c, gồm người đã biết mắc bệnh.
- `self_report_no_diabetes`: `DIQ010=2` — **chưa từng được bác sĩ báo mắc tiểu đường theo tự khai**, không chứng minh là không mắc bệnh.
- `screening_candidate`: như trên, loại **những ca có ghi nhận rõ `RIDEXPRG=1`**. `RIDEXPRG=NaN` vẫn được giữ lại vì biến này không công bố ở nhiều nhóm; **không được suy diễn họ chắc chắn không mang thai**.

Bảng sẽ giúp chọn quần thể **sau khi xem số mẫu và nhãn thực tế**.

In [4]:
self_report_no_diabetes = all_measured.loc[all_measured["DIQ010"].eq(2)].copy()
screening_candidate = self_report_no_diabetes.loc[~self_report_no_diabetes["RIDEXPRG"].eq(1)].copy()

cohorts = {
    "all_measured": all_measured,
    "self_report_no_diabetes": self_report_no_diabetes,
    "screening_candidate": screening_candidate,
}
rows = []
for name, frame in cohorts.items():
    positive = int(frame["a1c_ge_57_candidate"].sum())
    rows.append({
        "cohort": name,
        "n": len(frame),
        "a1c_ge57_n": positive,
        "a1c_lt57_n": len(frame) - positive,
        "a1c_ge57_pct_unweighted": round(100 * positive / len(frame), 2) if len(frame) else None,
        "bmi_missing_n": int(frame["BMXBMI"].isna().sum()),
        "bmi_missing_pct": round(frame["BMXBMI"].isna().mean() * 100, 2) if len(frame) else None,
        "pregnancy_code1_n": int(frame["RIDEXPRG"].eq(1).sum()),
        "pregnancy_status_na_n": int(frame["RIDEXPRG"].isna().sum()),
    })
cohort_summary = pd.DataFrame(rows)
display(cohort_summary)
print("Lưu ý: tỷ lệ label chỉ của mẫu quan sát, chưa tính survey weights.")

,cohort,n,a1c_ge57_n,a1c_lt57_n,a1c_ge57_pct_unweighted,bmi_missing_n,bmi_missing_pct,pregnancy_code1_n,pregnancy_status_na_n
0,all_measured,5767,2299,3468,39.86,81,1.40,37,4704
1,self_report_no_diabetes,4735,1391,3344,29.38,57,1.20,37,3724
2,screening_candidate,4698,1387,3311,29.52,57,1.21,0,3724


Lưu ý: tỷ lệ label chỉ của mẫu quan sát, chưa tính survey weights.


## 5. Đặc trưng ứng viên và kiểm tra thiếu dữ liệu

Baseline chỉ gồm `RIDAGEYR`, `RIAGENDR`, `BMXBMI`; có thể mở rộng huyết áp, hoạt động thể chất ở các tệp bổ sung sau. `LBXGH`, `DIQ010`, `DIQ160`, mã cá thể và survey weight **không phải** predictor.

Tại Bước 4 **KHÔNG được fit imputer/scaler**; để Bước 5 split train/test trước rồi mới fit pipeline trên train.

In [5]:
FEATURES = ["RIDAGEYR", "RIAGENDR", "BMXBMI"]
LABEL = "a1c_ge_57_candidate"
FORBIDDEN_PREDICTORS = {
    "LBXGH", "DIQ010", "DIQ160", "SEQN", "RIDEXPRG",
    "WTPH2YR", "WTMEC2YR", "SDMVSTRA", "SDMVPSU", "a1c_band_review"
}
assert not (set(FEATURES) & FORBIDDEN_PREDICTORS)
assert all(name in screening_candidate.columns for name in FEATURES + [LABEL])

feature_missing = (screening_candidate[FEATURES]
    .isna().agg(["sum", "mean"]).T.rename(columns={"sum":"missing_n", "mean":"missing_rate"}))
feature_missing["missing_pct"] = (feature_missing["missing_rate"] * 100).round(2)
display(feature_missing)
print("Giá trị giới RIAGENDR:")
display(screening_candidate["RIAGENDR"].value_counts(dropna=False).to_frame("n"))

,missing_n,missing_rate,missing_pct
RIDAGEYR,0.0,0.000000,0.00
RIAGENDR,0.0,0.000000,0.00
BMXBMI,57.0,0.012133,1.21


Giá trị giới RIAGENDR:


,n
RIAGENDR,
2.0,2614
1.0,2084


## 6. Kiểm tra các giá trị BMI/HbA1c đặc biệt, KHÔNG tự động xóa

Dùng phân vị 0,5% và 99,5% để **đánh dấu các giá trị đáng kiểm tra**, không dùng để loại mẫu hoặc fit clipping trên toàn bộ dữ liệu. Nếu huấn luyện sau này cần xử lý ngoại lệ, các ngưỡng học từ dữ liệu phải được ước lượng trên **training set**.

In [6]:
review_cols = ["BMXBMI", "LBXGH"]
quality_rows = []
for col in review_cols:
    observed = screening_candidate[col].dropna()
    if observed.empty:
        quality_rows.append({"column": col, "observed_n": 0})
        continue
    q005, q995 = observed.quantile([0.005, 0.995]).tolist()
    n_flagged = int(((observed < q005) | (observed > q995)).sum())
    quality_rows.append({
        "column": col, "observed_n": len(observed),
        "min": float(observed.min()), "p0_5": float(q005),
        "median": float(observed.median()), "p99_5": float(q995),
        "max": float(observed.max()), "review_flag_n": n_flagged,
    })
quality_review = pd.DataFrame(quality_rows)
display(quality_review)
print("Các quantile trên chỉ phục vụ review, không loại/khử outlier.")

,column,observed_n,min,p0_5,median,p99_5,max,review_flag_n
0,BMXBMI,4641,11.1,17.5,27.8,56.3,74.8,45
1,LBXGH,4698,3.2,4.4,5.4,7.5,17.1,45


Các quantile trên chỉ phục vụ review, không loại/khử outlier.


## 7. Xuất các file **ỨNG VIÊN CHỜ DUYỆT**

- `nhanes_step4_all_measured_REVIEW_ONLY.csv`: so sánh nhãn/tập đầy đủ, giữ cột cơ bản.
- `nhanes_step4_screening_candidate_REVIEW_ONLY.csv`: dữ liệu đã có nhãn ứng viên và **các đặc trưng baseline** cùng `SEQN` đối chiếu, **không chứa HbA1c gốc** để hạn chế leakage.
- Tệp thông tin `metadata` độc lập giữ một số thông tin cần kiểm tra nguồn/trọng số, **không nhập vào mô hình**.

Thư mục `data/raw/` giữ nguyên không thay đổi. Có thể tắt xuất CSV bằng `EXPORT_FILES = False`.

In [7]:
EXPORT_FILES = True
model_cols = ["SEQN"] + FEATURES + [LABEL]
all_review = all_measured[model_cols].copy()
screen_review = screening_candidate[model_cols].copy()
assert all(col not in screen_review.columns for col in ["LBXGH", "DIQ010", "DIQ160", "RIDEXPRG", "WTPH2YR"])
assert all_review["SEQN"].is_unique and screen_review["SEQN"].is_unique

if EXPORT_FILES:
    all_review.to_csv(PROCESSED_DIR / "nhanes_step4_all_measured_REVIEW_ONLY.csv", index=False)
    screen_review.to_csv(PROCESSED_DIR / "nhanes_step4_screening_candidate_REVIEW_ONLY.csv", index=False)
    # Metadata giữ riêng để kiểm tra trạng thái nhãn / thiết kế mẫu.
    meta_cols = ["SEQN", "LBXGH", "DIQ010", "RIDEXPRG", "WTPH2YR", "SDMVSTRA", "SDMVPSU"]
    screening_candidate[meta_cols].to_csv(OUT_DIR / "step4_screening_metadata_NOT_FEATURES.csv", index=False)
    cohort_summary.to_csv(OUT_DIR / "step4_cohort_summary.csv", index=False)
    feature_missing.to_csv(OUT_DIR / "step4_feature_missing.csv", index=True)
    quality_review.to_csv(OUT_DIR / "step4_values_to_review.csv", index=False)
    summary = {
        "source": "NHANES August 2021-August 2023",
        "status": "REVIEW_ONLY_NOT_TRAINED",
        "adults_n": int(len(adults)),
        "all_measured_n": int(len(all_measured)),
        "screening_candidate_n": int(len(screening_candidate)),
        "label_definition_candidate": "LBXGH >= 5.7; not diagnosis / future incidence",
        "features_candidate": FEATURES,
        "cohorts": cohort_summary.to_dict(orient="records"),
    }
    (OUT_DIR / "step4_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Đã xuất các CSV/JSON vào:", PROCESSED_DIR, "và", OUT_DIR)
else:
    print("EXPORT_FILES=False: chưa ghi file nào.")

Đã xuất các CSV/JSON vào: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\processed và C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\preprocessing


## 8. Output tổng kết — **GỬI LẠI CHO TÔI**

Gửi toàn bộ phần in dưới đây kèm bảng `step4_cohort_summary.csv` (nếu thuận tiện). Tôi sẽ dùng kết quả thật để chốt quần thể cho Bước 5. **Không thực hiện huấn luyện ở notebook này.**

In [8]:
print("===== BƯỚC 4 — NHANES DATA PREPROCESSING (người dùng chạy) =====")
print("Python:", sys.version.split()[0], "| Pandas:", pd.__version__)
print("Adults >=20:", len(adults))
print("Có HbA1c:", len(all_measured))
print("Có đồng thời HbA1c + BMI:", int(adults[["LBXGH", "BMXBMI"]].notna().all(axis=1).sum()))
print("--- COHORT SUMMARY ---")
print(cohort_summary.to_string(index=False))
print("--- FEATURE MISSING (screening_candidate) ---")
print(feature_missing[["missing_n", "missing_pct"]].to_string())
print("--- VALUES FLAGGED FOR REVIEW ---")
print(quality_review.to_string(index=False))
print("FEATURES candidate:", FEATURES)
print("LABEL candidate:", LABEL)
print("TRAINED MODEL: NONE | FIT PREPROCESSOR: NONE | TRAIN/TEST SPLIT: NONE")
print("===== KẾT THÚC BƯỚC 4 — CHỜ RÀ SOÁT =====")

===== BƯỚC 4 — NHANES DATA PREPROCESSING (người dùng chạy) =====
Python: 3.13.9 | Pandas: 3.0.0
Adults >=20: 7809
Có HbA1c: 5767
Có đồng thời HbA1c + BMI: 5686
--- COHORT SUMMARY ---
                 cohort    n  a1c_ge57_n  a1c_lt57_n  a1c_ge57_pct_unweighted  bmi_missing_n  bmi_missing_pct  pregnancy_code1_n  pregnancy_status_na_n
           all_measured 5767        2299        3468                    39.86             81             1.40                 37                   4704
self_report_no_diabetes 4735        1391        3344                    29.38             57             1.20                 37                   3724
    screening_candidate 4698        1387        3311                    29.52             57             1.21                  0                   3724
--- FEATURE MISSING (screening_candidate) ---
          missing_n  missing_pct
RIDAGEYR        0.0         0.00
RIAGENDR        0.0         0.00
BMXBMI         57.0         1.21
--- VALUES FLAGGED FOR REVIEW -

## 9. Dừng tại đây

Không chạy model training / train-test split / imputation vào lúc này. Sau khi tôi xem output của bạn, chúng ta mới thống nhất:

1. Tập sàng lọc nào là chính và cách xử lý ca đã được chẩn đoán;
2. Có giữ lại mã giới tính, tuổi và BMI làm baseline hay cần bổ sung huyết áp, hành vi từ tệp bổ sung;
3. Trường hợp BMI thiếu nên xử lý ra sao **sau khi split train/test**;
4. Cách tính metric, trọng số và giới hạn suy rộng quần thể.

**Bước 5 sẽ có phần huấn luyện thực tế. Tôi sẽ báo trước và bạn tự chạy các cell train.**